# Hierarchical Time-Series Forecasting — Buffalo Milk Yield

Consolidated forecasting notebook (secondary / off-proposal track). Merges the useful,
de-duplicated code from `HTS.ipynb`, `HTS_Forecasting.ipynb`, `Hierarchical.ipynb`, and
`Hierchical_Structure.ipynb` into one ordered workflow. Original notebooks are kept in `raw/`.

**Hierarchy:** `Total → Farm → Animal`  ·  **Target:** `milk_kg` (test-day, resampled monthly)

- **Part A — Reconciled forecasting:** aggregate → base forecasts (AutoETS / SeasonalNaive)
  → reconciliation (BottomUp, MinTrace) → evaluation → cross-validation → plots.
- **Part B — Wood's lactation-curve tensors:** DIM-grid snapping + robust Wood's-curve fit per
  animal → `milk`/`protein` curve tensors + fit-quality report.
- **Appendix — series retrieval helpers.**

*Dropped from the originals as superseded scratch:* manual `lil_matrix`/`coo_matrix` S-matrix
builds (replaced by `aggregate()`), chunked long-panel builders, and the exploratory
RandomForest feature-importance / `mixedlm` experiments.

# Part A — Reconciled hierarchical forecasting

## A.0  Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# path = '/content/drive/MyDrive/Thesis_Data/'   # Colab
path = '../../Thesis_Data/'                        # Local

DATA_FILE = path + 'Final_Data/Final_Merged_Data.csv'
OUT_DIR   = path + 'HTS_Results/'

# ── Modelling choices ─────────────────────────────────────────────────────────
TARGET    = 'milk_kg'
FREQ      = 'MS'        # Month-Start — test-day records are ~monthly
H         = 3           # forecast horizon (months ahead)
N_WINDOWS = 3           # rolling cross-validation windows
MIN_OBS   = 12          # min monthly obs required per bottom-level series

# ── Hierarchy bottom level ────────────────────────────────────────────────────
# 'farm'   → Total > Farm (~300 series)           ← default, memory-safe
# 'animal' → Total > Farm > Animal (~80k series)  ← needs large RAM (>32 GB)
BOTTOM_LEVEL = 'farm'

SEED = 42

## A.1  Install & import

In [ ]:
%%capture
!pip install hierarchicalforecast statsforecast

In [ ]:
import os, gc, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, Naive, SeasonalNaive

from hierarchicalforecast.utils import aggregate
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace
from hierarchicalforecast.evaluation import HierarchicalEvaluation

warnings.filterwarnings('ignore')
os.makedirs(OUT_DIR, exist_ok=True)
print('Libraries loaded')

## A.2  Load data

In [ ]:
df = pd.read_csv(
    DATA_FILE,
    usecols=['Farm_Code', 'Animal_ID', 'dtt', TARGET],
    parse_dates=['dtt'],
)
df['Farm_Code'] = df['Farm_Code'].astype(str)
df['Animal_ID'] = df['Animal_ID'].astype(str)

print(f'Records   : {len(df):,}')
print(f'Animals   : {df["Animal_ID"].nunique():,}')
print(f'Farms     : {df["Farm_Code"].nunique():,}')
print(f'Date range: {df["dtt"].min().date()} -> {df["dtt"].max().date()}')
df.head()

## A.3  Resample to monthly frequency

Test-day records are irregular (~one per month per animal). Snap each record to the first day
of its month (`MS`) and take the **mean** when multiple tests fall in the same month.

In [ ]:
df['ds'] = df['dtt'].dt.to_period('M').dt.to_timestamp()

df_monthly = (
    df.groupby(['Farm_Code', 'Animal_ID', 'ds'])[TARGET]
      .mean().reset_index()
      .rename(columns={TARGET: 'y'})
)
print(f'Monthly records (animal level): {len(df_monthly):,}')
print(f'Date range: {df_monthly["ds"].min().date()} -> {df_monthly["ds"].max().date()}')
df_monthly.head()

## A.4  Build the hierarchical panel (`Y_df`), summing matrix (`S_df`), and `tags`

`hierarchicalforecast.utils.aggregate` builds all three at once and keeps the S-matrix sparse.

| `BOTTOM_LEVEL` | Levels | Series count | RAM |
|---|---|---|---|
| `'farm'`   | Total → Farm          | ~300 | < 1 GB |
| `'animal'` | Total → Farm → Animal | ~80k | > 32 GB |

In [ ]:
df_hier = df_monthly.copy()
df_hier['Total'] = 'Total'

if BOTTOM_LEVEL == 'farm':
    spec = [['Total'], ['Total', 'Farm_Code']]
    df_agg = df_hier.groupby(['Total', 'Farm_Code', 'ds'])['y'].sum().reset_index()

elif BOTTOM_LEVEL == 'animal':
    # Make Animal unique across farms by prefixing Farm_Code
    df_hier['Animal'] = df_hier['Farm_Code'] + '/' + df_hier['Animal_ID']
    spec = [['Total'], ['Total', 'Farm_Code'], ['Total', 'Farm_Code', 'Animal']]
    df_agg = df_hier[['Total', 'Farm_Code', 'Animal', 'ds', 'y']].copy()
else:
    raise ValueError(f"BOTTOM_LEVEL must be 'farm' or 'animal', got: {BOTTOM_LEVEL}")

Y_df, S_df, tags = aggregate(df_agg, spec)
Y_df = Y_df.reset_index(drop=True)

all_ids    = Y_df['unique_id'].unique().tolist()
bottom_ids = tags[list(tags.keys())[-1]]     # last level = bottom
farm_ids   = tags.get('Farm_Code', [])

print(f'BOTTOM_LEVEL : {BOTTOM_LEVEL}')
print(f'Total series : {Y_df["unique_id"].nunique():,}')
for level, ids in tags.items():
    print(f'  {level:<12}: {len(ids):,} series')
print(f'S_df shape   : {S_df.shape}')
print(f'Y_df rows    : {len(Y_df):,}')
print(f'Date range   : {Y_df["ds"].min().date()} -> {Y_df["ds"].max().date()}')

## A.5  Verify coherence

At any date, `Total` must equal the sum of the bottom-level series.

In [ ]:
check_date = sorted(Y_df['ds'].unique())[len(Y_df['ds'].unique()) // 2]
snap = Y_df[Y_df['ds'] == check_date].set_index('unique_id')['y']
total_val  = snap.reindex(['Total']).sum()
bottom_sum = snap.reindex(bottom_ids).sum()
print(f'Coherence check at {pd.Timestamp(check_date).date()}:')
print(f'  Total series value   : {total_val:,.2f}')
print(f'  Sum of bottom series : {bottom_sum:,.2f}')
print(f'  Coherent             : {np.isclose(total_val, bottom_sum)}')

## A.6  Visualise the hierarchy

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

tot = Y_df[Y_df['unique_id'] == 'Total'].sort_values('ds')
axes[0].plot(tot['ds'], tot['y'], color='#1a1a2e', linewidth=2)
axes[0].set_title('Total Herd — Monthly Milk Yield', fontsize=13, fontweight='bold')
axes[0].set_ylabel('kg')

np.random.seed(SEED)
sample_farms = np.random.choice(farm_ids, min(5, len(farm_ids)), replace=False)
for fid in sample_farms:
    ts = Y_df[Y_df['unique_id'] == fid].sort_values('ds')
    axes[1].plot(ts['ds'], ts['y'], label=str(fid).replace('Farm/', ''), linewidth=1)
axes[1].set_title('Sample of Farm-Level Series (5 farms)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('kg'); axes[1].legend(fontsize=7, ncol=3)

if BOTTOM_LEVEL == 'animal':
    farm_code_sample = str(sample_farms[0]).replace('Farm/', '')
    animals_in_farm = [b for b in bottom_ids if b.startswith(farm_code_sample + '/')]
    for aid in np.random.choice(animals_in_farm, min(5, len(animals_in_farm)), replace=False):
        ts = Y_df[Y_df['unique_id'] == aid].sort_values('ds')
        axes[2].plot(ts['ds'], ts['y'], linewidth=1, alpha=0.8)
    axes[2].set_title(f'Sample Animals from Farm {farm_code_sample}', fontsize=13, fontweight='bold')
axes[2].set_ylabel('kg')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(OUT_DIR + 'hierarchy_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## A.7  Filter: keep only series with enough history

StatsForecast needs a minimum number of observations per series. Drop bottom-level series with
fewer than `MIN_OBS` months, then re-run `aggregate` on the filtered panel to stay coherent.

In [ ]:
obs_counts = (
    Y_df[Y_df['unique_id'].isin(bottom_ids)]
    .groupby('unique_id')['ds'].count()
)
valid_bottom = obs_counts[obs_counts >= MIN_OBS].index.tolist()
print(f'Bottom series with >= {MIN_OBS} months : {len(valid_bottom):,}  '
      f'(dropped {len(bottom_ids) - len(valid_bottom):,})')

if BOTTOM_LEVEL == 'farm':
    df_agg_filtered = df_agg[df_agg['Farm_Code'].isin(set(valid_bottom))].copy()
else:
    df_agg_filtered = df_agg[df_agg['Animal'].isin(set(valid_bottom))].copy()

Y_df, S_df, tags = aggregate(df_agg_filtered, spec)
Y_df = Y_df.reset_index(drop=True)
all_ids    = Y_df['unique_id'].unique().tolist()
bottom_ids = tags[list(tags.keys())[-1]]
farm_ids   = tags.get('Farm_Code', [])
print(f'After filter -> {Y_df["unique_id"].nunique():,} series, {len(Y_df):,} rows')

# Persist the coherent panel for reuse
Y_df.to_parquet(OUT_DIR + 'Y_df.parquet', index=False)
S_df.to_parquet(OUT_DIR + 'S_df.parquet')
with open(OUT_DIR + 'tags.json', 'w') as f:
    json.dump({k: list(v) for k, v in tags.items()}, f, indent=2)
gc.collect()

## A.8  Train / test split

In [ ]:
cutoff  = Y_df['ds'].max() - pd.DateOffset(months=H)
Y_train = Y_df[Y_df['ds'] <= cutoff].copy()
Y_test  = Y_df[Y_df['ds'] >  cutoff].copy()
print(f'Train: up to {cutoff.date()}  ({Y_train["ds"].nunique()} months)')
print(f'Test : {Y_test["ds"].min().date()} -> {Y_test["ds"].max().date()}  ({Y_test["ds"].nunique()} months)')

## A.9  Base forecasts

Fit **AutoETS** and a strong **SeasonalNaive(12)** baseline independently on every series.
`AutoARIMA` is available but slower — uncomment if time allows.

In [ ]:
models = [
    AutoETS(season_length=12),
    SeasonalNaive(season_length=12),
    # AutoARIMA(season_length=12),
]
sf = StatsForecast(
    models=models, freq=FREQ, n_jobs=-1,
    fallback_model=SeasonalNaive(season_length=12),
)
print(f'Fitting {Y_train["unique_id"].nunique():,} series...')
Y_hat_df    = sf.forecast(df=Y_train, h=H, fitted=True)
Y_fitted_df = sf.forecast_fitted_values()
Y_hat_df.to_parquet(OUT_DIR + 'Y_hat_base.parquet', index=False)
print('Base forecasts done'); Y_hat_df.head()

## A.10  Reconciliation

Raw base forecasts are **incoherent** (Total ≠ sum of Farm). Reconciliation enforces coherence.

| Method | Description |
|---|---|
| **BottomUp** | Aggregate bottom forecasts up. |
| **MinTrace(ols)** | OLS optimal reconciliation (Wickramasuriya 2019). |
| **MinTrace(mint_shrink)** | MinTrace with shrinkage covariance — often best. |

In [ ]:
reconcilers = [
    BottomUp(),
    MinTrace(method='ols'),
    MinTrace(method='mint_shrink'),
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df, Y_df=Y_train, S=S_df, tags=tags)
Y_rec_df.to_parquet(OUT_DIR + 'Y_reconciled.parquet', index=False)
print('Reconciliation done'); print(Y_rec_df.columns.tolist()); Y_rec_df.head()

## A.11  Evaluation

RMSE and MAE at each level of the hierarchy, benchmarked against SeasonalNaive.

In [ ]:
def rmse(y_true, y_pred): return np.sqrt(np.mean((y_true - y_pred) ** 2))
def mae(y_true, y_pred):  return np.mean(np.abs(y_true - y_pred))

evaluator = HierarchicalEvaluation(evaluators=[rmse, mae])
eval_results = evaluator.evaluate(
    Y_hat=Y_rec_df, Y_test=Y_test, tags=tags, benchmark='SeasonalNaive',
)
print('=== EVALUATION RESULTS ==='); print(eval_results.to_string())
eval_results.to_csv(OUT_DIR + 'evaluation_results.csv')

## A.12  Rolling cross-validation

In [ ]:
Y_cv_df = sf.cross_validation(df=Y_df, h=H, n_windows=N_WINDOWS, step_size=H)

model_cols = [c for c in Y_cv_df.columns if c not in ['unique_id', 'ds', 'cutoff', 'y']]
results_cv = []
for level_name, level_ids in tags.items():
    sub = Y_cv_df[Y_cv_df['unique_id'].isin(level_ids)]
    for col in model_cols:
        r = np.sqrt(np.mean((sub['y'] - sub[col]) ** 2))
        m = np.mean(np.abs(sub['y'] - sub[col]))
        results_cv.append({'Level': level_name, 'Model': col, 'RMSE': round(r, 3), 'MAE': round(m, 3)})
cv_table = pd.DataFrame(results_cv).pivot(index='Level', columns='Model', values='RMSE')
print('=== CV RMSE per Level ==='); print(cv_table.to_string())
cv_table.to_csv(OUT_DIR + 'cv_rmse_per_level.csv')

## A.13  Plot reconciled vs actual

In [ ]:
def plot_forecast(unique_id, Y_train, Y_test, Y_rec_df, model_col, ax, title=None):
    train = Y_train[Y_train['unique_id'] == unique_id].sort_values('ds').tail(24)
    test  = Y_test[Y_test['unique_id'] == unique_id].sort_values('ds')
    fc    = Y_rec_df[Y_rec_df['unique_id'] == unique_id].sort_values('ds')
    ax.plot(train['ds'], train['y'], color='steelblue', label='Train', linewidth=1.5)
    ax.plot(test['ds'], test['y'], color='black', label='Actual', linewidth=1.5, linestyle='--')
    if model_col in fc.columns:
        ax.plot(fc['ds'], fc[model_col], color='tomato', label=model_col, linewidth=1.5)
    if len(test):
        ax.axvline(test['ds'].min(), color='grey', linestyle=':', linewidth=1)
    ax.set_title(title or unique_id, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7); ax.set_ylabel('milk_kg')

reconciled_col = [c for c in Y_rec_df.columns if 'mint_shrink' in c and 'AutoETS' in c]
reconciled_col = reconciled_col[0] if reconciled_col else Y_rec_df.columns[-1]
print(f'Plotting column: {reconciled_col}')

np.random.seed(SEED)
plot_farm = list(np.random.choice(farm_ids, min(2, len(farm_ids)), replace=False))
panels = [('Total', 'Total Herd')] + [(f, f'Farm: {f}') for f in plot_farm]

fig, axes = plt.subplots(len(panels), 1, figsize=(14, 4 * len(panels)))
axes = np.atleast_1d(axes)
for ax, (uid, title) in zip(axes, panels):
    plot_forecast(uid, Y_train, Y_test, Y_rec_df, reconciled_col, ax, title)
plt.tight_layout()
plt.savefig(OUT_DIR + 'reconciled_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part B — Wood's lactation-curve interpolation & tensors

Independent of reconciliation: fit a robust **Wood's curve** `y(t) = a·t^b·e^(−c·t)` to each
animal's lactation, on a 12-point 30-day DIM grid, producing dense `milk` / `protein` curve
tensors (one row per animal) plus a fit-quality report. Useful as curve-shape features
(see the clustering folder) and for smoothing sparse test-day series.

## B.1  Load full records and snap DIM to a monthly grid

In [ ]:
import numpy as np, pandas as pd
from itertools import product

df = pd.read_csv(path + 'Final_Data/Final_Merged_Data.csv')

# Monthly grid: midpoints of 30-day windows → [15, 45, ..., 345] (12 points, 0–365 DIM)
DIM_GRID = np.arange(15, 365, 30)

def snap_to_grid(dim_value, grid=DIM_GRID):
    return grid[np.argmin(np.abs(grid - dim_value))]

df['DIM_grid'] = df['DIM'].apply(snap_to_grid)

# Resolve duplicates (two tests in the same window) by mean
df_grid = (
    df.groupby(['Farm_Code', 'Animal_ID', 'DIM_grid'])[['milk_kg', 'protein_p']]
      .mean().reset_index()
)
print(f'Records after grid snapping: {len(df_grid):,}  ·  animals: {df_grid["Animal_ID"].nunique():,}')

In [ ]:
# Complete skeleton: every animal × every grid point, then merge actual values (NaN where missing)
all_animals = df_grid[['Farm_Code', 'Animal_ID']].drop_duplicates()
grid_df = pd.DataFrame(
    [(r.Farm_Code, r.Animal_ID, d) for _, r in all_animals.iterrows() for d in DIM_GRID],
    columns=['Farm_Code', 'Animal_ID', 'DIM_grid'],
)
df_full = grid_df.merge(df_grid, on=['Farm_Code', 'Animal_ID', 'DIM_grid'], how='left')
print(f'Full grid: {df_full.shape}  ·  missing milk_kg: {df_full["milk_kg"].isna().mean()*100:.1f}%')

## B.2  Robust Wood's-curve fit per animal → curve tensors

In [ ]:
from scipy.optimize import curve_fit, OptimizeWarning
from scipy.stats import zscore
import warnings
warnings.filterwarnings('ignore', category=OptimizeWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

def woods_curve(dim, a, b, c):
    return a * np.power(dim, b) * np.exp(-c * dim)

def fit_woods_robust(animal_df, target_col, grid=DIM_GRID):
    observed = animal_df.dropna(subset=[target_col])
    if len(observed) < 3:
        return pd.Series(np.nan, index=grid)
    dim_obs, y_obs = observed['DIM_grid'].values, observed[target_col].values
    if len(y_obs) >= 4:                       # reject outliers (|z| > 2.5)
        mask = np.abs(zscore(y_obs)) < 2.5
        if mask.sum() >= 3:
            dim_obs, y_obs = dim_obs[mask], y_obs[mask]
    try:
        params, _ = curve_fit(woods_curve, dim_obs, y_obs,
                              p0=[y_obs.mean(), 0.1, 0.003],
                              bounds=([0, 0, 0], [np.inf, 2, 0.1]), maxfev=5000)
        return pd.Series(np.clip(woods_curve(grid, *params), 0, None), index=grid)
    except Exception:
        return pd.Series(np.interp(grid, dim_obs, y_obs), index=grid)

results_milk, results_protein = [], []
grouped = df_full.groupby(['Farm_Code', 'Animal_ID'])
for i, ((farm, animal), grp) in enumerate(grouped):
    if i % 5000 == 0:
        print(f'Processing {i:,}/{len(grouped):,}...')
    mp = fit_woods_robust(grp, 'milk_kg')
    pp = fit_woods_robust(grp, 'protein_p')
    results_milk.append({'Farm_Code': farm, 'Animal_ID': animal, **{f'dim_{d}': v for d, v in mp.items()}})
    results_protein.append({'Farm_Code': farm, 'Animal_ID': animal, **{f'dim_{d}': v for d, v in pp.items()}})

df_milk    = pd.DataFrame(results_milk)
df_protein = pd.DataFrame(results_protein)
print(f'Milk tensor: {df_milk.shape}  ·  Protein tensor: {df_protein.shape}')

In [ ]:
# Drop biologically impossible curves, then persist the tensors
dim_cols   = [c for c in df_milk.columns if c.startswith('dim_')]
valid_both = (df_milk[dim_cols] > 0.1).all(axis=1) & (df_protein[dim_cols] > 0.01).all(axis=1)
df_milk    = df_milk[valid_both].reset_index(drop=True)
df_protein = df_protein[valid_both].reset_index(drop=True)
print(f'Animals kept: {len(df_milk):,}  ·  dropped: {(~valid_both).sum():,}')
df_milk.to_parquet('milk_tensor.parquet', index=False)
df_protein.to_parquet('protein_tensor.parquet', index=False)

## B.3  Fit-quality report (Wood's parameters + R²/RMSE/MAE per animal)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

fit_results = []
for (farm, animal), grp in df_full.groupby(['Farm_Code', 'Animal_ID']):
    for target in ['milk_kg', 'protein_p']:
        observed = grp.dropna(subset=[target])
        if len(observed) < 3:
            continue
        dim_obs, y_obs = observed['DIM_grid'].values, observed[target].values
        if len(y_obs) >= 4:
            mask = np.abs(zscore(y_obs)) < 2.5
            if mask.sum() >= 3:
                dim_obs, y_obs = dim_obs[mask], y_obs[mask]
        try:
            params, _ = curve_fit(woods_curve, dim_obs, y_obs, p0=[y_obs.mean(), 0.1, 0.003],
                                  bounds=([0, 0, 0], [np.inf, 2, 0.1]), maxfev=5000)
            y_pred = np.clip(woods_curve(dim_obs, *params), 0, None)
            fit_results.append({'Farm_Code': farm, 'Animal_ID': animal, 'target': target,
                                'n_obs': len(y_obs), 'a': params[0], 'b': params[1], 'c': params[2],
                                'R2': r2_score(y_obs, y_pred),
                                'RMSE': np.sqrt(mean_squared_error(y_obs, y_pred)),
                                'MAE': mean_absolute_error(y_obs, y_pred), 'fit_method': 'woods'})
        except Exception:
            fit_results.append({'Farm_Code': farm, 'Animal_ID': animal, 'target': target,
                                'n_obs': len(y_obs), 'a': np.nan, 'b': np.nan, 'c': np.nan,
                                'R2': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'fit_method': 'linear_fallback'})

df_fits = pd.DataFrame(fit_results)
df_fits['fit_quality'] = pd.cut(df_fits['R2'], bins=[-np.inf, 0, 0.5, 0.8, 1.0],
                                labels=['poor', 'weak', 'acceptable', 'good'])
df_fits.to_parquet('woods_fit_quality.parquet', index=False)

print("=== Wood's fit quality (median) ===")
print(f"{'Metric':<12}{'milk_kg':>12}{'protein_p':>12}")
for metric, col in [('R2', 'R2'), ('RMSE', 'RMSE'), ('MAE', 'MAE')]:
    mk = df_fits[df_fits['target'] == 'milk_kg'][col].median()
    pr = df_fits[df_fits['target'] == 'protein_p'][col].median()
    print(f'{metric:<12}{mk:>12.3f}{pr:>12.3f}')
print()
print(df_fits.groupby(['target', 'fit_quality']).size().unstack(fill_value=0))

## B.4  Sanity plot: observed vs Wood's-interpolated curves

In [ ]:
grid_points = [int(c.split('_')[1]) for c in dim_cols]
sample_animals = df_milk.sample(6, random_state=100)['Animal_ID'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, animal_id in zip(axes.flatten(), sample_animals):
    row = df_milk[df_milk['Animal_ID'] == animal_id].iloc[0]
    observed = df_grid[df_grid['Animal_ID'] == animal_id][['DIM_grid', 'milk_kg']].dropna()
    ax.plot(grid_points, row[dim_cols].values, color='steelblue', linewidth=1.8,
            linestyle='--', label='Interpolated')
    ax.scatter(observed['DIM_grid'], observed['milk_kg'], color='crimson', zorder=5, s=60, label='Observed')
    ax.set_title(f'Animal: {animal_id}', fontsize=9)
    ax.set_xlabel('DIM (days)'); ax.set_ylabel('milk_kg'); ax.legend(fontsize=8)
plt.suptitle("Lactation Curves: Observed vs Wood's Interpolated", fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
# Appendix — series retrieval helpers

In [ ]:
# Convenience accessors for the long HTS panel (Y_df with columns: unique_id, ds, y)
def get_animal(hts_df, farm_code, animal_id):
    sid = f"{farm_code}/{animal_id}"
    return hts_df[hts_df['unique_id'] == sid].sort_values('ds').reset_index(drop=True)

def get_farm(hts_df, farm_code):
    return hts_df[hts_df['unique_id'] == f'Farm/{farm_code}'].sort_values('ds').reset_index(drop=True)

def list_farms(hts_df):
    return [s.replace('Farm/', '') for s in hts_df['unique_id'].unique() if str(s).startswith('Farm/')]

def list_animals(hts_df, farm_code):
    prefix = f'{farm_code}/'
    return [s.replace(prefix, '') for s in hts_df['unique_id'].unique() if str(s).startswith(prefix)]

# Example:
# print(list_farms(Y_df)[:5])
# get_animal(Y_df, '521513', 'IT003990094623')